In [1]:
# load data from npy files
import numpy as np
input = np.load('./initial_data/function_1/initial_inputs.npy')
output = np.load('./initial_data/function_1/initial_outputs.npy')

print(input.shape)
print(input[:10])

print(output.shape)
print(output[:10])

(10, 2)
[[0.31940389 0.76295937]
 [0.57432921 0.8798981 ]
 [0.73102363 0.73299988]
 [0.84035342 0.26473161]
 [0.65011406 0.68152635]
 [0.41043714 0.1475543 ]
 [0.31269116 0.07872278]
 [0.68341817 0.86105746]
 [0.08250725 0.40348751]
 [0.88388983 0.58225397]]
(10,)
[ 1.32267704e-079  1.03307824e-046  7.71087511e-016  3.34177101e-124
 -3.60606264e-003 -2.15924904e-054 -2.08909327e-091  2.53500115e-040
  3.60677119e-081  6.22985647e-048]


In [4]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# -----------------------------------------------------------
# 1. Define the search space (bounds for each of the 2 dims)
# -----------------------------------------------------------
# Replace these with the real min/max for your problem
bounds = np.array([
    [0.0, 1.0],  # bounds for dimension 0
    [0.0, 1.0],  # bounds for dimension 1
])

# -----------------------------------------------------------
# 2. Fit a Gaussian Process on your existing data
#    Assumes you already have:
#       input  -> shape (n_samples, 2)
#       output -> shape (n_samples,) or (n_samples, 1)
# -----------------------------------------------------------

# Make sure output is 1D
y = np.asarray(output).ravel()
X = np.asarray(input)

kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(length_scale=0.5, nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,
    normalize_y=True,
    random_state=0,
)

gp.fit(X, y)

# -----------------------------------------------------------
# 3. Define Expected Improvement (EI) acquisition function
# -----------------------------------------------------------

def expected_improvement(X_candidates, gp, y_best, xi=0.01):
    """
    X_candidates: array of shape (n_candidates, 2)
    gp: fitted GaussianProcessRegressor
    y_best: current best observed value
    xi: exploration parameter (small >0 encourages exploration)
    """
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)

    # Avoid division by zero
    sigma = np.maximum(sigma, 1e-9)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0

    return ei.ravel()

# -----------------------------------------------------------
# 4. Propose the next point by maximizing EI
#    (here we use random search over the box defined by `bounds`)
# -----------------------------------------------------------

def sample_random_points(bounds, n_points, random_state=None):
    rng = np.random.default_rng(random_state)
    dim = bounds.shape[0]
    return rng.uniform(bounds[:, 0], bounds[:, 1], size=(n_points, dim))

def propose_next_point(input, output, bounds, n_candidates=10000, xi=0.01, random_state=0):
    X = np.asarray(input)
    y = np.asarray(output).ravel()

    # Fit a new GP (you can reuse the earlier one if you like)
    kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(length_scale=0.5, nu=2.5) \
             + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=5,
        normalize_y=True,
        random_state=random_state,
    )
    gp.fit(X, y)

    y_best = y.max()

    # Sample candidate points in the search space
    X_candidates = sample_random_points(bounds, n_candidates, random_state=random_state)

    # Compute EI over candidates
    ei = expected_improvement(X_candidates, gp, y_best, xi=xi)

    # Choose the candidate with the highest EI
    best_idx = np.argmax(ei)
    x_next = X_candidates[best_idx]

    return x_next, ei[best_idx], gp

# -----------------------------------------------------------
# 5. Actually get the next point to sample
# -----------------------------------------------------------

x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.63696169 0.26978671]
EI at that point: 5.52441292692546e-26


/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [12]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from scipy.stats import norm

# -----------------------------------------------------------
# Generic utilities (work for any dimension D)
# -----------------------------------------------------------

def sample_random_points(bounds, n_points, random_state=None):
    """
    Sample n_points uniformly inside a hyper-rectangle.
    
    bounds: array of shape (D, 2), where D = dimensionality
            bounds[i, 0] = lower bound for dim i
            bounds[i, 1] = upper bound for dim i
    """
    rng = np.random.default_rng(random_state)
    bounds = np.asarray(bounds)
    D = bounds.shape[0]
    return rng.uniform(bounds[:, 0], bounds[:, 1], size=(n_points, D))


def expected_improvement(X_candidates, gp, y_best, xi=0.01):
    """
    Expected Improvement acquisition function.
    
    X_candidates: (n_candidates, D)
    gp: fitted GaussianProcessRegressor
    y_best: best observed target value so far
    xi: exploration parameter
    """
    mu, sigma = gp.predict(X_candidates, return_std=True)
    mu = mu.reshape(-1, 1)
    sigma = sigma.reshape(-1, 1)

    # avoid division by zero
    sigma = np.maximum(sigma, 1e-9)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma == 0.0] = 0.0

    return ei.ravel()


def fit_gp_model(X, y, random_state=0):
    """
    Fit a Gaussian Process model to the data.
    Works for any X.shape = (n_samples, D).
    """
    kernel = ConstantKernel(1.0, (0.1, 10.0)) * Matern(length_scale=0.5, nu=2.5) \
             + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e-1))

    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=5,
        normalize_y=True,
        random_state=random_state,
    )
    gp.fit(X, y)
    return gp


def propose_next_point(input, output, bounds, n_candidates=10000, xi=0.01, random_state=0):
    """
    Propose the next point to sample by maximizing Expected Improvement.
    
    input:   array of shape (n_samples, D)
    output:  array of shape (n_samples,) or (n_samples, 1)
    bounds:  array of shape (D, 2)
    """
    X = np.asarray(input)
    y = np.asarray(output).ravel()
    
    # Fit GP on current data
    gp = fit_gp_model(X, y, random_state=random_state)
    y_best = y.max()

    # Sample candidate points in D-dimensional space
    X_candidates = sample_random_points(bounds, n_candidates, random_state=random_state)

    # Compute EI and pick the best
    ei = expected_improvement(X_candidates, gp, y_best, xi=xi)
    best_idx = np.argmax(ei)

    x_next = X_candidates[best_idx]
    ei_value = ei[best_idx]
    return x_next, ei_value, gp

# -----------------------------------------------------------
# Example usage for 3D, 4D, ..., 8D
# -----------------------------------------------------------

# Assume you already have:
#   input  -> shape (n_samples, D)
#   output -> shape (n_samples,) or (n_samples, 1)

# D = input.shape[1]
# You must define bounds with shape (D, 2)

# Example: 5D problem (generalizes to 3D–8D)
# bounds = np.array([
#     [0.0, 1.0],   # dim 0
#     [-1.0, 2.0],  # dim 1
#     [10.0, 20.0], # dim 2
#     [0.0, 5.0],   # dim 3
#     [100.0, 200.0]# dim 4
# ])

# Get next point to evaluate:
# x_next, ei_value, gp_model = propose_next_point(input, output, bounds)
# print("Next point to sample:", x_next)
# print("EI at that point:", ei_value)

def infer_bounds_from_data(input, pad_ratio=0.05):
    """
    Infer bounds from the existing samples.
    pad_ratio = % margin added beyond min/max (default 5%).
    """
    X = np.asarray(input)
    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    
    # Add padding to allow exploration
    padding = (maxs - mins) * pad_ratio
    mins -= padding
    maxs += padding
    
    return np.vstack([mins, maxs]).T   # shape (D, 2)

In [6]:
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.63696169 0.26978671]
EI at that point: 5.52441292692546e-26


/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [7]:
input = np.load('./initial_data/function_2/initial_inputs.npy')
output = np.load('./initial_data/function_2/initial_outputs.npy')

In [8]:
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.77536717 0.93522844]
EI at that point: 0.03965901262113387


In [10]:
input = np.load('./initial_data/function_3/initial_inputs.npy')
output = np.load('./initial_data/function_3/initial_outputs.npy')

In [13]:
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.36227252 0.32987054 0.46428246]
EI at that point: 0.018389453339689815


In [14]:
input = np.load('./initial_data/function_4/initial_inputs.npy')
output = np.load('./initial_data/function_4/initial_outputs.npy')
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.40656703 0.41361442 0.37804933 0.45540797]
EI at that point: 2.082849742543928


/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [15]:
input = np.load('./initial_data/function_5/initial_inputs.npy')
output = np.load('./initial_data/function_5/initial_outputs.npy')
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.33484391 0.76737484 0.90168002 0.95212843]
EI at that point: 22.541957552734253


/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified upper bound 0.1. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [16]:
input = np.load('./initial_data/function_6/initial_inputs.npy')
output = np.load('./initial_data/function_6/initial_outputs.npy')
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.43681772 0.38354373 0.55462841 0.79732752 0.13701335]
EI at that point: 0.18065552221524028


In [17]:
input = np.load('./initial_data/function_7/initial_inputs.npy')
output = np.load('./initial_data/function_7/initial_outputs.npy')
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.18079117 0.5616356  0.26256344 0.09244324 0.35095803 0.75775324]
EI at that point: 0.004393804455067103


In [18]:
input = np.load('./initial_data/function_8/initial_inputs.npy')
output = np.load('./initial_data/function_8/initial_outputs.npy')
bounds = infer_bounds_from_data(input)
x_next, ei_value, gp_model = propose_next_point(input, output, bounds)

print("Next point to sample:", x_next)
print("EI at that point:", ei_value)

Next point to sample: [0.02460228 0.25798831 0.05712641 0.31943214 0.81896624 0.33659617
 0.18221378 0.54653135]
EI at that point: 0.5631586620743


/Users/roychan/Documents/personal/study/imperial/ml/capstone1/.venv/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 10.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


In [ ]:
input.shape


(40, 8)